In [80]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

In [65]:
URL = "https://www.ocair.com/travelers/flights/arrivals-departures/"
API_URL = "https://s3-us-west-2.amazonaws.com/files.ocair.com/data/sna_export.js"
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Referer": "https://www.ocair.com/",
    "Origin": "https://www.ocair.com",
}

response = requests.get(API_URL, headers=HEADERS, timeout=20)
response.raise_for_status()
# Python -> sends HTTP request -> website server
# Server -> returns html page
print("Status code", response.status_code)
print("Content Type", response.headers.get("Content-Type"))
#200 = success
#404 = page not found
#403 = blocked

Status code 200
Content Type application/json


In [66]:
data = response.json()
print("Top-level keys:")
for key in data.keys():
    print("-", key)

Top-level keys:
- link
- datetime
- provider
- airlines
- airports
- flights


In [77]:
arrivals = data.get("flights").get("arrivals")
dept = data.get("flights").get("departures")
print(arrivals)
print(dept)

[{'id': '708312', 'status': 'Arrived', 'gate': '20', 'Claim': '5', 'times': {'scheduled': '2026-03-13 07:55:00', 'actual': '2026-03-13 07:40:00'}, 'codes': ['WN4455'], 'route': ['SJC']}, {'id': '708324', 'status': 'Arrived', 'gate': '18', 'Claim': '5', 'times': {'scheduled': '2026-03-13 08:00:00', 'actual': '2026-03-13 07:53:00'}, 'codes': ['WN4115'], 'route': ['SMF']}, {'id': '-1', 'status': 'Arrived', 'gate': '4', 'Claim': '2', 'times': {'scheduled': '2026-03-13 08:05:00', 'actual': '2026-03-13 07:43:00'}, 'codes': ['MX602'], 'route': ['PVU']}, {'id': '708330', 'status': 'Arrived', 'gate': '17', 'Claim': '5', 'times': {'scheduled': '2026-03-13 08:10:00', 'actual': '2026-03-13 07:56:00'}, 'codes': ['WN4097'], 'route': ['OAK']}, {'id': '708331', 'status': 'Arrived', 'gate': '19', 'Claim': '5', 'times': {'scheduled': '2026-03-13 08:15:00', 'actual': '2026-03-13 08:22:00'}, 'codes': ['WN3280'], 'route': ['LAS']}, {'id': '708339', 'status': 'Arrived', 'gate': '22B', 'Claim': 'N/A', 'times

In [ ]:
## Collect arrival and dept rows for data analysis ###
arrival_rows = []
for flight in arrivals:
    city = "NAN"
    if flight.get("route") != None:
        code = flight.get('route')
        ### route containse the airport code ###
        code = code[0]
        city = data["airports"][code]
    else:
        code = "NAN"
    arrival_rows.append({
        "id": flight.get("id"),
        "city": city,
        "status": flight.get("status"),
        "gate": flight.get("gate"),
        "claim": flight.get("Claim"),
        "scheduled_time": flight.get("times", {}).get("scheduled"),
        "actual_time": flight.get("times", {}).get("actual"),
        "flight_codes": ", ".join(flight.get("codes", [])),
        "arrival_airport_code": ", ".join(flight.get("route", [])),
    })

dept_rows = []
for flight in dept:
    city = "NAN"
    if flight.get("route") != None:
        code = flight.get('route')
        ### route containse the airport code ###
        code = code[0]
        city = data["airports"][code]
    else:
        code = "NAN"
    dept_rows.append({
        "id": flight.get("id"),
        "city": city,
        "status": flight.get("status"),
        "gate": flight.get("gate"),
        "claim": flight.get("Claim"),
        "scheduled_time": flight.get("times", {}).get("scheduled"),
        "actual_time": flight.get("times", {}).get("actual"),
        "flight_codes": ", ".join(flight.get("codes", [])),
        "arrival_airport_code": ", ".join(flight.get("route", [])),
    })

In [84]:
new_data = arrival_rows + dept_rows
df = pd.DataFrame(new_data)
#Get todays date
today = datetime.today().strftime("%Y-%m-%d")

path = "../data/"
filename = f"JohnWayneFlights_{today}.csv"

df.to_csv(path + filename, index=False)